# PART 05 · 2장 작업형 2 핵심 패턴

교재 428~448쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 전체의 분석 흐름

#### 2. 공통 라이브러리와 데이터 불러오기

공통 라이브러리와 세 문제의 데이터를 한 번에 불러온다.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score
binary_path = 'data/part03_type2/binary_learning_renewal/'
multi_path = 'data/part03_type2/multiclass_ticket_priority/'
reg_path = 'data/part03_type2/regression_delivery_minutes/'
binary_train = pd.read_csv(binary_path+'train.csv')
binary_test = pd.read_csv(binary_path+'test.csv')
binary_sample = pd.read_csv(binary_path+'sample_submission.csv')
multi_train = pd.read_csv(multi_path+'train.csv')
multi_test = pd.read_csv(multi_path+'test.csv')
multi_sample = pd.read_csv(multi_path+'sample_submission.csv')
reg_train = pd.read_csv(reg_path+'train.csv')
reg_test = pd.read_csv(reg_path+'test.csv')
reg_sample = pd.read_csv(reg_path+'sample_submission.csv')
print(binary_train.shape, binary_test.shape, binary_sample.shape)
print(multi_train.shape, multi_test.shape, multi_sample.shape)
print(reg_train.shape, reg_test.shape, reg_sample.shape)

(720, 15) (240, 14) (240, 1)
(780, 15) (260, 14) (260, 1)
(840, 15) (280, 14) (280, 1)


## 01 작업형 2 핵심 패턴

## 02 이진분류 제출 패턴

### 1 문제와 지표 확인

#### 1. 이진분류 문제 확인하기

train에는 target인 renew_next_month가 포함되어 있고 test에는 없다. target 값의 종류와 개수를 확인하면 이진분류인지 판단할 수 있다.

In [2]:
print(binary_train['renew_next_month'].value_counts().sort_index())
print('target_mean: ', round(binary_train['renew_next_month'].mean(), 4))

renew_next_month
0    332
1    388
Name: count, dtype: int64
target_mean:  0.5389


#### 2. 평가 지표와 제출값 연결하기

### 2 제출 흐름과 확인

#### 1. 이진분류 전체 제출 흐름

검증용 데이터는 먼저 학습용과 검증용으로 나눈다. 결측치 대체값과 인코딩 컬럼은 학습용 데이터에서 정하고, 검증용 데이터에는 같은 기준을 적용한다. 모델을 선택한 뒤에는 전체 train으로 전처리 기준과 모델을 다시 학습해 test를 예측한다.

In [3]:
target_col = 'renew_next_month'
id_col = 'student_id'
binary_X_raw = binary_train.drop(columns=[target_col, id_col]).copy()
binary_y = binary_train[target_col].copy()
binary_test_raw = binary_test.drop(columns=[id_col]).copy()
binary_X_train_raw, binary_X_valid_raw, binary_y_train, binary_y_valid = train_test_split(
    binary_X_raw,
    binary_y,
    test_size=0.2,
    random_state=0,
    stratify=binary_y
)
def prepare_features(reference_X, other_X):
    reference_X = reference_X.copy()
    other_X = other_X.copy()
    cat_cols = reference_X.select_dtypes(include='object').columns
    num_cols = reference_X.select_dtypes(exclude='object').columns
    for col in num_cols:
        median_value = reference_X[col].median()
        reference_X[col] = reference_X[col].fillna(median_value)
        other_X[col] = other_X[col].fillna(median_value)
    for col in cat_cols:
        reference_X[col] = reference_X[col].fillna('unknown')
        other_X[col] = other_X[col].fillna('unknown')
    reference_X = pd.get_dummies(reference_X)
    other_X = pd.get_dummies(other_X)
    other_X = other_X.reindex(columns=reference_X.columns, fill_value=0)
    return reference_X, other_X
binary_X_train, binary_X_valid = prepare_features(
    binary_X_train_raw, binary_X_valid_raw)
binary_model = RandomForestClassifier(n_estimators=200, random_state=0)
binary_model.fit(binary_X_train, binary_y_train)
positive_index = list(binary_model.classes_).index(1)
binary_valid_pred = binary_model.predict_proba(
    binary_X_valid
)[:, positive_index]
binary_score = roc_auc_score(binary_y_valid, binary_valid_pred)
# 모델과 전처리 방식을 확정한 뒤 전체 train으로 최종 학습한다.
binary_X_full, binary_test_encoded = prepare_features(
    binary_X_raw, binary_test_raw
)
binary_final_model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
binary_final_model.fit(binary_X_full, binary_y)
final_positive_index = list(binary_final_model.classes_).index(1)
binary_test_pred = binary_final_model.predict_proba(
    binary_test_encoded
)[:, final_positive_index]
binary_result = binary_sample.copy()
binary_result['pred'] = binary_test_pred
binary_result.to_csv('result.csv', index=False)
print('submission_shape: ', binary_result.shape)
print('submission_columns: ', binary_result.columns.tolist())

submission_shape:  (240, 1)
submission_columns:  ['pred']


#### 2. 결과 해석과 제출 확인

## 03 다중분류 제출 패턴

### 1 문제 확인과 제출 흐름

#### 1. 다중분류 문제 확인하기

다중분류인지 target의 클래스 수부터 확인해본다.

In [4]:
print(multi_train['priority_level'].value_counts().sort_index())
print('class_count: ', multi_train['priority_level'].nunique())

priority_level
0    304
1    359
2    117
Name: count, dtype: int64
class_count:  3


#### 2. 다중분류 전체 제출 흐름

### 2 지표 확인과 제출

#### 1. Accuracy와 Macro F1

#### 2. 제출 파일 생성

#### 3. Macro F1 검증과 전체 재학습

다중분류에서 클래스별 성능을 같은 비중으로 확인하려면 f1_score( )에 average=‘macro’를 지정한다. 검증 단계에서 모델을 정한 뒤 전체 train으로 다시 학습하고, 클래스 제출 문제에서는 predict( ) 결과를 저장한다.

In [5]:
multi_target = 'priority_level'
multi_id = 'ticket_id'
multi_X_raw = multi_train.drop(
    columns=[multi_target, multi_id]
).copy()
multi_y = multi_train[multi_target].copy()
multi_test_raw = multi_test.drop(columns=[multi_id]).copy()
(
    multi_X_train_raw,
    multi_X_valid_raw,
    multi_y_train,
    multi_y_valid
) = train_test_split(
    multi_X_raw,
    multi_y,
    test_size=0.2,
    random_state=0,
    stratify=multi_y
)
multi_X_train, multi_X_valid = prepare_features(
    multi_X_train_raw, multi_X_valid_raw
)
multi_model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
multi_model.fit(multi_X_train, multi_y_train)
multi_valid_pred = multi_model.predict(multi_X_valid)
multi_accuracy = accuracy_score(
    multi_y_valid, multi_valid_pred
)
multi_macro_f1 = f1_score(
    multi_y_valid,
    multi_valid_pred,
    average='macro'
)

전체 train으로 다시 학습해 제출값을 만들고 검증 점수와 함께 확인해본다.

In [6]:
multi_X_full, multi_test_encoded = prepare_features(
    multi_X_raw, multi_test_raw
)
multi_final_model = RandomForestClassifier(
    n_estimators=200,
    random_state=0
)
multi_final_model.fit(multi_X_full, multi_y)
multi_test_pred = multi_final_model.predict(
    multi_test_encoded
)
multi_result = multi_sample.copy()
multi_result['pred'] = multi_test_pred
multi_result.to_csv('result.csv', index=False)
print('accuracy: ', round(multi_accuracy, 4))
print('macro_f1: ', round(multi_macro_f1, 4))
print('submission_shape: ', multi_result.shape)
print(multi_result['pred'].value_counts().sort_index())
print('missing: ', multi_result.isna().sum().sum())

accuracy:  0.5897
macro_f1:  0.4693
submission_shape:  (260, 1)
pred
0    106
1    145
2      9
Name: count, dtype: int64
missing:  0


## 04 회귀 제출 패턴

### 1 문제 확인과 제출 흐름

#### 1. 회귀 문제 확인하기

회귀 문제인지 target의 분포부터 확인해본다.

In [7]:
print(reg_train['delivery_minutes'].describe().round(4))

count    840.0000
mean      53.3377
std       13.6003
min       21.6000
25%       43.6000
50%       52.9000
75%       61.2000
max      100.5000
Name: delivery_minutes, dtype: float64


#### 2. 회귀 문제 제출 흐름

### 2 지표 확인과 제출

#### 1. 회귀 평가지표 확인하기

#### 2. 제출 파일 생성

#### 3. target 점검과 MAE 검증

회귀 target에서 0이나 음수가 비정상값으로 정의되었다면 학습 전에 해당 행을 제외한다. 어떤 값이 비정상인지는 문제의 컬럼 설명을 기준으로 판단한다.

다음 보조 예제처럼 조건식으로 유효한 target만 남길 수 있다.

In [8]:
target_example = pd.Series([52.1, 0.0, -3.0, 47.5])
valid_example = target_example[target_example > 0]
print(valid_example.tolist())
#[52.1,47.5]

[52.1, 47.5]


MAE로 검증 점수를 확인한 뒤 전체 train으로 다시 학습하고 test의 연속형 예측값을 제출한다.

In [9]:
reg_target = 'delivery_minutes'
reg_id = 'order_id'
invalid_target = reg_train[reg_target] <= 0
print('invalid_target_count: ', invalid_target.sum())
reg_work = reg_train.loc[~invalid_target].copy()
reg_X_raw = reg_work.drop(
    columns=[reg_target, reg_id]
).copy()
reg_y = reg_work[reg_target].copy()
reg_test_raw = reg_test.drop(columns=[reg_id]).copy()
(
    reg_X_train_raw,
    reg_X_valid_raw,
    reg_y_train,
    reg_y_valid
) = train_test_split(
    reg_X_raw,
    reg_y,
    test_size=0.2,
    random_state=0
)
reg_X_train, reg_X_valid = prepare_features(
    reg_X_train_raw, reg_X_valid_raw
)
reg_model = RandomForestRegressor(
    n_estimators=200,
    random_state=0
)
reg_model.fit(reg_X_train, reg_y_train)
reg_valid_pred = reg_model.predict(reg_X_valid)
reg_mae = mean_absolute_error(
    reg_y_valid, reg_valid_pred
)

invalid_target_count:  0


전체 train으로 다시 학습해 회귀 제출값을 만들고 범위까지 점검해본다.

In [10]:
reg_X_full, reg_test_encoded = prepare_features(
    reg_X_raw, reg_test_raw
)
reg_final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=0
)
reg_final_model.fit(reg_X_full, reg_y)
reg_test_pred = reg_final_model.predict(
    reg_test_encoded
)
reg_result = reg_sample.copy()
reg_result['pred'] = reg_test_pred
reg_result.to_csv('result.csv', index=False)
print('mae: ', round(reg_mae, 4))
print('submission_shape: ', reg_result.shape)
print('pred_min: ', round(reg_result['pred'].min(), 4))
print('pred_max: ', round(reg_result['pred'].max(), 4))
print('missing: ', reg_result.isna().sum().sum())

mae:  7.0827
submission_shape:  (280, 1)
pred_min:  33.461
pred_max:  87.2915
missing:  0


## 05 target 분리 패턴

### 1 target 찾기와 분리

#### 1. target 컬럼 찾기

문제 설명에서 target 컬럼명을 먼저 확인한다. 이름이 명확하지 않다면 train에는 있고 test에는 없는 컬럼을 비교할 수 있다. 단, train과 test의 구조가 다른 이유가 항상 target 하나뿐인 것은 아니므로 문제 설명과 함께 확인한다.

In [11]:
train_only = set(binary_train.columns) - set(binary_test.columns)
print(train_only)

{'renew_next_month'}


#### 2. X와 y 분리하기

target 컬럼을 y로 분리하고 X에서 빠졌는지 확인해본다.

In [12]:
target_col = 'renew_next_month'
binary_y = binary_train[target_col]
binary_X_raw = binary_train.drop(columns=[target_col])
print('y_shape: ', binary_y.shape)
print('X_shape: ', binary_X_raw.shape)
print('target_in_X: ', target_col in binary_X_raw.columns)

y_shape:  (720,)
X_shape:  (720, 14)
target_in_X:  False


### 2 제외와 누수 방지

#### 1. ID 컬럼 제외하기

student_id는 각 행을 구분하는 식별자다. ID 숫자의 크기 자체가 수강 연장 여부를 설명하는 의미 있는 값은 아니므로 모델 입력에서 제외한다. 제출 파일에 ID가 필요하다면 별도로 보관한다.

In [13]:
binary_X_raw = binary_X_raw.drop(columns=['student_id'])
binary_test_raw = binary_test.drop(columns=['student_id'])
print(binary_X_raw.shape)
print(binary_test_raw.shape)

(720, 13)
(240, 13)


#### 2. 데이터 누수 방지하기

## 06 인코딩 패턴

### 1 인코딩 준비

#### 1. 인코딩 대상 찾기

식별자와 target을 제외한 뒤 인코딩 대상인 문자형 컬럼을 찾아본다.

In [14]:
binary_X_raw = binary_train.drop(
    columns=['renew_next_month', 'student_id']
).copy()
binary_test_raw = binary_test.drop(columns=['student_id']).copy()
cat_cols = binary_X_raw.select_dtypes(include='object').columns
num_cols = binary_X_raw.select_dtypes(exclude='object').columns
print('categorical_columns: ', cat_cols.tolist())
print('categorical_count: ', len(cat_cols))

categorical_columns:  ['age_group', 'region', 'device_type', 'course_category']
categorical_count:  4


#### 2. 결측치 처리하기

수치형 결측치는 train에서 계산한 중앙값으로 채운다. 범주형 결측치는 unknown이라는 새 범주로 채운다. test의 결측치에도 train에서 정한 기준을 그대로 적용한다.

In [15]:
for col in num_cols:
    median_value = binary_X_raw[col].median()
    binary_X_raw[col] = binary_X_raw[col].fillna(median_value)
    binary_test_raw[col] = binary_test_raw[col].fillna(median_value)
for col in cat_cols:
    binary_X_raw[col] = binary_X_raw[col].fillna('unknown')
    binary_test_raw[col] = binary_test_raw[col].fillna('unknown')
print(binary_X_raw.isna().sum().sum())
print(binary_test_raw.isna().sum().sum())

0
0


### 2 더미변수와 기준 적용

#### 1. 더미변수 만들기

#### 2. 학습 데이터 기준으로 인코딩하기

train과 test에 각각 get_dummies를 적용하고 train 기준으로 컬럼을 맞춰 본다.

In [16]:
binary_X_encoded = pd.get_dummies(binary_X_raw)
binary_test_encoded = pd.get_dummies(binary_test_raw)
binary_test_encoded = binary_test_encoded.reindex(
    columns=binary_X_encoded.columns,
    fill_value=0
)
print('X_encoded: ', binary_X_encoded.shape)
print('test_encoded: ', binary_test_encoded.shape)
print(
    'same_columns: ',
    binary_X_encoded.columns.equals(binary_test_encoded.columns)
)

X_encoded:  (720, 29)
test_encoded:  (240, 29)
same_columns:  True


## 07 컬럼 불일치 처리 패턴

### 1 불일치 원인과 확인

#### 1. 컬럼 불일치가 발생하는 이유

#### 2. 서로 다른 컬럼 확인하기

두 데이터를 따로 인코딩하면 컬럼이 달라질 수 있다. 어느 쪽에만 있는 컬럼인지 확인해본다.

In [17]:
X_separate = pd.get_dummies(binary_X_raw)
test_separate = pd.get_dummies(binary_test_raw)
train_only = sorted(
    set(X_separate.columns) - set(test_separate.columns)
)
test_only = sorted(
    set(test_separate.columns) - set(X_separate.columns)
)
print('X_shape: ', X_separate.shape)
print('test_shape: ', test_separate.shape)
print('train_only: ', train_only)
print('test_only: ', test_only)

X_shape:  (720, 29)
test_shape:  (240, 31)
train_only:  []
test_only:  ['course_category_클라우드', 'device_type_smart_tv']


### 2 컬럼 맞추기와 검증

#### 1. reindex( )로 컬럼 맞추기

reindex( )로 test의 컬럼을 train 기준에 맞춰 본다.

In [18]:
test_aligned = test_separate.reindex(
    columns=X_separate.columns,
    fill_value=0
)
print(test_aligned.shape)
print(X_separate.columns.equals(test_aligned.columns))

(240, 29)
True


#### 2. 최종 컬럼 검증하기

컬럼 개수만 같다고 충분하지 않다. 이름과 순서가 모두 같은지 columns.equals( )로 확인한다. 결과가 True이면 학습과 예측에 사용할 입력 구조가 일치한다.

> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
test_encoded = test_encoded.reindex(
    columns=X_encoded.columns,
    fill_value=0
)
```

## 08 predict_proba 사용 패턴

### 1 두 메서드 비교

#### 1. predict( )와 predict_proba( ) 비교

#### 2. 확률 결과 구조 확인하기

predict_proba( ) 결과의 구조를 확인해본다.

In [19]:
proba = binary_model.predict_proba(binary_X_valid)
print('classes: ', binary_model.classes_)
print('proba_shape: ', proba.shape)
print('row_sum: ', proba[0].sum())

classes:  [0 1]
proba_shape:  (144, 2)
row_sum:  1.0


### 2 클래스와 확률 선택

#### 1. 클래스 순서 확인하기

#### 2. 양성 클래스(라벨 1) 확률 선택하기

classes_에서 라벨 1의 위치를 찾아 해당 열의 확률만 선택한다.

In [20]:
positive_index = list(binary_model.classes_).index(1)
binary_valid_pred = proba[:, positive_index]
final_positive_index = list(binary_final_model.classes_).index(1)
binary_test_pred = binary_final_model.predict_proba(
    binary_test_encoded
)[:, final_positive_index]

## 09 result.csv 생성 패턴

### 1 제출 기준과 생성

#### 1. 제출 파일 기준 확인하기

#### 2. sample_submission을 복사해 만들기

sample_submission을 복사해 pred 컬럼에 예측값을 넣어 본다.

In [21]:
binary_result = binary_sample.copy()
binary_result['pred'] = binary_test_pred
print(binary_result.shape)
print(binary_result.columns.tolist())

(240, 1)
['pred']


### 2 저장과 검증

#### 1. 인덱스 없이 저장하기

저장할 때는 index=False를 지정한다.

In [22]:
binary_result.to_csv('result.csv', index=False)

#### 2. 저장한 파일 다시 읽어 검증하기

저장한 파일을 다시 읽어 구조를 검증해본다.

In [23]:
check = pd.read_csv('result.csv')
print('shape: ', check.shape)
print('columns: ', check.columns.tolist())
print('missing: ', check.isna().sum().sum())

shape:  (240, 1)
columns:  ['pred']
missing:  0


> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
best_setting: B
0    112
1    116
2     32
Name: count, dtype: int64
```